# Character-level tokenizer and vocabulary

We're going to implement a character-level tokenization scheme, suitable for taking a corpus of text and outputting a vocabulary over that text. 

Our Implementation will consist of three classes:

(1) A Tokenizer base class, with the following functions exposed to the user:
* __init__(self, filename: str), which takes a filename for a file containing text. Subclasses might take additional options at initialization. 
* build_tokens(self), which opens the files and using the text creates a set of tokens
* get_tokens(self), which returns a set of tokens
* encode(self, text: str), which given some text splits it into a list of tokens

(2) A CharTokenizer class, which inherits from Tokenizer. Here the tokens is a list of characters, including all ASCII characters and any additional characters found in the text. (Optionally, I might also implement a UnicodeTokenizer which can encode any Unicode character by storing the bytes.)

(3) A Vocab class, which is designed to be used with any subclass of Tokenizer. The following functions are exposed:
* __init__(self, tokenizer) 
* __getitem__(self, idx): converts an index or list or indexes to a token or list of tokens
* token_to_idx(self, tokens): converts a token or list of tokens to an index
* str_to_idx(self, tokens): converts a string to a list of indices

In [16]:
from abc import ABC, abstractmethod
from collections.abc import Iterable

In [17]:
class Tokenizer(ABC):
    @abstractmethod
    def build_tokens(self):
        """Creates a set of tokens"""
        pass

    @property
    def tokens(self):
        return self._tokens

    @abstractmethod
    def encode(self, text):
        pass

In [18]:
class CharTokenizer(Tokenizer):
    """A character-level tokenizer"""
    def build_tokens(self, **kwargs):
        # use ASCII characters
        ascii_nums = [i for i in range(32, 255) if i not in {127, 129, 141, 143, 144, 157}] + [10]
        ascii_chars = [chr(num) for num in ascii_nums]
        self._tokens = set(['<unk>']) | set(ascii_chars)
    
    def encode(self, text):
        return [char if char in self._tokens else '<unk>' for char in text]


In [19]:
tokenizer = CharTokenizer()
tokenizer.build_tokens()
print(list(tokenizer.encode(f"hello world my name is nick now let's insert some gibberish {chr(314)} {chr(240)} {chr(257)}\nline 2")))

['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', ' ', 'm', 'y', ' ', 'n', 'a', 'm', 'e', ' ', 'i', 's', ' ', 'n', 'i', 'c', 'k', ' ', 'n', 'o', 'w', ' ', 'l', 'e', 't', "'", 's', ' ', 'i', 'n', 's', 'e', 'r', 't', ' ', 's', 'o', 'm', 'e', ' ', 'g', 'i', 'b', 'b', 'e', 'r', 'i', 's', 'h', ' ', '<unk>', ' ', 'ð', ' ', '<unk>', '\n', 'l', 'i', 'n', 'e', ' ', '2']


In [20]:
chr(160)

'\xa0'

In [24]:
class Vocab:
    def __init__(self, tokenizer, filename=None):
        self.tokenizer = tokenizer
        self.tokenizer.build_tokens(filename=filename)
        self.build_vocab(self.tokenizer.tokens)

    def build_vocab(self, tokens):
        self._token_to_idx = {}
        self._idx_to_token = {}

        for i, token in enumerate(tokens):
            self._token_to_idx[token] = i
            self._idx_to_token[i] = token
    
    def __len__(self):
        return len(self._token_to_idx)

    def __getitem__(self, idx):
        if isinstance(idx, Iterable):
            return [self.__getitem__(i) for i in idx]
        else:
            return self._idx_to_token[idx]
        
    def token_to_idx(self, token):
        if isinstance(token, list):
            return [self.token_to_idx(t) for t in token]
        else:
            if token not in self._token_to_idx.keys():
                return self._token_to_idx['<unk>']
            return self._token_to_idx[token]
    
    def encode(self, text):
        return [self.token_to_idx(token) for token in self.tokenizer.encode(text)]


In [28]:
tokenizer = CharTokenizer()
vocab = Vocab(tokenizer, None)


print("Length of vocab is ", len(vocab))
N = 219
print(f"The first {N} vocab tokens are: {[vocab[i] for i in range(N)]}")

print(vocab.token_to_idx('a'))

tokens = vocab.encode(f"hello world my name is nick I am {chr(257)} feet tall")

print(vocab[tokens])

Length of vocab is  219
The first 219 vocab tokens are: ['õ', 'É', 'y', 'X', '.', 'Ó', '{', 'Ì', '6', 'Ú', 'Á', '²', 'Ã', 'V', 'Å', 'Æ', '`', '\x8c', '¶', '¬', 'Ï', 'ï', 'À', 'O', '^', 'K', 'J', '\x80', '\x95', 'Í', 'Ø', '\x9c', 'ó', 'a', '³', '>', '¹', 'h', '¨', '¯', 'r', 'R', 'ì', '/', '\x82', 'Õ', 'm', '\x8e', 'c', 'N', 'à', '!', 'e', 'ý', '3', '¿', 'È', '4', ']', 'Q', 'P', 'æ', '~', 'ø', '\x88', '\x9b', '2', '\x9a', ':', 's', 'p', '\x93', 'Ñ', 'ä', 'S', '+', 'n', 'Ç', '}', 'ü', 'B', '\x84', '|', '#', 'T', 'v', 'E', 'Ù', 'ë', '½', '°', '5', 'A', '×', ' ', 'å', '\x92', '¦', 'w', 'U', '\x83', '\xad', 'â', 'ª', '1', 'Ë', 'Ò', '\\', 'M', 'û', '&', 'i', 'ú', 'H', '\x8a', '\x96', '\x8b', ')', 'ù', '%', '©', '0', '8', 't', '=', 'ô', '\x94', '»', '@', 'd', '"', 'D', 'f', '[', 'Ü', 'W', 'j', 'ß', 'á', 'Z', 'l', 'Þ', '\x97', '\x85', '«', 'Ê', '_', 'ò', 'I', 'x', ';', 'é', '\x99', 'Ô', 'G', '¥', 'ã', '-', '<', 'ð', ',', 'Ä', '7', 'C', '9', 'ç', 'L', 'Ý', "'", 'º', '\xa0', 'ñ', 'o', '¡', 'u', '